# Topic 12 — Solutions: Capstone Investigation

*Reference solutions for the objective tasks. Try the practice first.* The Warm-Up concept questions, Interview Lens and Reflection have **no key** — by design.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
master = (items.merge(products[['product_id','category','unit_cost']], on='product_id', how='left')
              .merge(orders[['order_id','channel','status','month','order_date']], on='order_id', how='left'))
master['line_cogs'] = master['quantity']*master['unit_cost']
master['line_profit'] = master['line_revenue'] - master['line_cogs']
returns = pd.read_csv(RAW+'returns.csv', dtype={'order_id':str})


### Warm-Up — NumPy recall (self-check)

In [ ]:
rev = np.array([100.,200.,0.,50.]); cogs = np.array([60.,120.,0.,25.])
mask = rev>0
margin = (rev[mask]-cogs[mask]).sum()/rev[mask].sum()
print(round(float(margin),4))

### Core lesson tasks

In [ ]:
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
print(len(master) == len(items))

In [ ]:
gross = master['line_revenue'].sum(); total_refunds = returns['refund_amount'].sum(); net = gross-total_refunds
n_orders = orders['order_id'].nunique(); aov = gross/n_orders
returns_rate = (orders['status']=='returned').mean()
gross_margin = (gross - master['line_cogs'].sum())/gross
print(f'gross £{gross:,.0f} net £{net:,.0f} AOV £{aov:.2f} returns {returns_rate:.1%} margin {gross_margin:.1%}')

In [ ]:
monthly = master.dropna(subset=['order_date']).set_index('order_date').sort_index()['line_revenue'].resample('ME').sum()
print(monthly.pct_change().describe())  # seasonal swings, not a structural decline

### Mixed review (earlier topics)

In [ ]:
tickets = pd.read_csv(RAW+'support_tickets.csv', dtype={'ticket_id':str})
m = tickets['message'].str.lower()
print(m.str.contains('late|where is', regex=True, na=False).sum(), m.str.contains('damaged|broken|leak', regex=True, na=False).sum())

In [ ]:
pivot = master.pivot_table(index='channel', columns='month', values='line_revenue', aggfunc='sum', margins=True, fill_value=0)
print(pivot.iloc[:, :3])

### Data detective

In [ ]:
print('Objective gate passed; the narrative (exec summary, findings, 3 numbered recommendations) is yours to defend.')

### Interview Lens — discussion points (not full answers)
- **Q30:** show lineage raw→clean→join→aggregate with row-count asserts and coverage of exclusions; reconcile order vs line level.
- **Q32:** accuracy, completeness, consistency, validity, uniqueness, timeliness — made concrete on Aurora (channel casing, unique customers, valid prices, parsed dates, matched keys, documented exclusions).